In [0]:
from pyspark.sql.functions import col, when, lit, current_timestamp, to_timestamp, trim, upper
from delta.tables import DeltaTable

# 1. Leemos la capa Bronze cruda
df_bronze = spark.read.table("logistics_prod.bronze.logistics_raw")

# 2. Aplicamos reglas de Calidad de Datos (Data Quality)
df_validated = df_bronze.withColumn(
    "dq_error_reason",
    when(col("Asset_ID").isNull() | (trim(col("Asset_ID")) == ""), lit("MISSING_ASSET_ID"))
    .when(col("Timestamp").isNull(), lit("MISSING_TIMESTAMP"))
    .when((col("Temperature").cast("double") < -50) | (col("Temperature").cast("double") > 60), lit("TEMPERATURE_OUT_OF_RANGE"))
    .when(col("Latitude").isNull() | col("Longitude").isNull(), lit("INVALID_COORDINATES"))
    .otherwise(lit(None))
)

# 3. Aislamos los registros defectuosos en la tabla de Cuarentena
df_quarantine = (
    df_validated.filter(col("dq_error_reason").isNotNull())
    .select("Timestamp", "Asset_ID", "Temperature", "Shipment_Status", "dq_error_reason", "_source_file")
    .withColumn("quarantined_at", current_timestamp())
)

df_quarantine.write.mode("overwrite").saveAsTable("logistics_prod.silver.logistics_quarantine")

# 4. Limpiamos y estandarizamos los registros válidos (snake_case y tipos exactos)
df_clean = (
    df_validated.filter(col("dq_error_reason").isNull())
    .select(
        to_timestamp(col("Timestamp")).alias("event_timestamp"),
        upper(trim(col("Asset_ID"))).alias("asset_id"),
        col("Latitude").cast("double").alias("latitude"),
        col("Longitude").cast("double").alias("longitude"),
        col("Inventory_Level").cast("int").alias("inventory_level"),
        trim(col("Shipment_Status")).alias("shipment_status"),
        col("Temperature").cast("double").alias("temperature_c"),
        col("Humidity").cast("double").alias("humidity_pct"),
        trim(col("Traffic_Status")).alias("traffic_status"),
        col("Waiting_Time").cast("int").alias("waiting_time_mins"),
        col("User_Transaction_Amount").cast("double").alias("transaction_amount"),
        col("User_Purchase_Frequency").cast("int").alias("purchase_frequency"),
        when(col("Logistics_Delay_Reason").isNull() | (trim(col("Logistics_Delay_Reason")) == "None"), lit("No Delay"))
            .otherwise(trim(col("Logistics_Delay_Reason"))).alias("delay_reason"),
        col("Asset_Utilization").cast("double").alias("asset_utilization_pct"),
        col("Demand_Forecast").cast("int").alias("demand_forecast"),
        col("Logistics_Delay").cast("int").alias("is_delayed"),
        current_timestamp().alias("processed_at")
    )
    .dropDuplicates(["asset_id", "event_timestamp"])
)

# 5. MERGE Idempotente en Silver
silver_table = "logistics_prod.silver.logistics_clean"

if not spark.catalog.tableExists(silver_table):
    df_clean.write.format("delta").saveAsTable(silver_table)
else:
    delta_silver = DeltaTable.forName(spark, silver_table)
    (
        delta_silver.alias("target")
        .merge(
            df_clean.alias("source"),
            "target.asset_id = source.asset_id AND target.event_timestamp = source.event_timestamp"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

print(f"Capa Silver lista. Filas limpias: {spark.table(silver_table).count()} | En cuarentena: {spark.table('logistics_prod.silver.logistics_quarantine').count()}")


# Emitimos el conteo de filas en cuarentena hacia el Lakeflow Job (Task Values)
quarantine_count = spark.table("logistics_prod.silver.logistics_quarantine").count()
clean_count = spark.table(silver_table).count()

dbutils.jobs.taskValues.set(key="quarantine_rows", value=quarantine_count)
dbutils.jobs.taskValues.set(key="clean_rows", value=clean_count)
print(f"Métricas publicadas al Job -> Clean: {clean_count} | Quarantine: {quarantine_count}")
